In [1]:
"""
Confirmatory Factor Analysis: GRMSAAW Scale Validation
=======================================================
Analyst  : Mintay Misgano
Tool     : Python 3 | semopy, pandas, matplotlib, seaborn
Dataset  : dfGRMSAAW.csv — simulated from Keum et al. (2018)
           published factor loadings and item descriptives.
           GRMSAAW — Gendered Racial Microaggressions Scale for
           Asian American Women (N=304, 22 items, 4 subscales)

Replicates the R lavaan CFA workflow using semopy, which shares
lavaan-compatible model syntax. Compares:
  Model 1 — Unidimensional (all 22 items on one factor)
  Model 2 — First-order correlated four-factor model

Fit indices: χ², CFI, GFI, RMSEA, SRMR, AIC, BIC
"""

'\nConfirmatory Factor Analysis: GRMSAAW Scale Validation\n=======================================================\nAnalyst  : Mintay Misgano\nTool     : Python 3 | semopy, pandas, matplotlib, seaborn\nDataset  : dfGRMSAAW.csv — simulated from Keum et al. (2018)\n           published factor loadings and item descriptives.\n           GRMSAAW — Gendered Racial Microaggressions Scale for\n           Asian American Women (N=304, 22 items, 4 subscales)\n\nReplicates the R lavaan CFA workflow using semopy, which shares\nlavaan-compatible model syntax. Compares:\n  Model 1 — Unidimensional (all 22 items on one factor)\n  Model 2 — First-order correlated four-factor model\n\nFit indices: χ², CFI, GFI, RMSEA, SRMR, AIC, BIC\n'

In [2]:
# =============================================================================
# 1. IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend (script / notebook execution)
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy.stats import chi2 as chi2_dist
import warnings
warnings.filterwarnings("ignore")

In [3]:
try:
    import semopy
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "semopy", "-q"])
    import semopy

In [4]:
print(f"semopy  : {semopy.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

semopy  : 2.3.11
pandas  : 2.3.3
numpy   : 2.2.6


In [5]:
# =============================================================================
# 2. DATA LOADING
#    Data were pre-simulated in R using MASS::mvrnorm with empirical=TRUE
#    from published loadings (Keum et al., 2018, Table 2) and item
#    descriptives (Table 4).  See CFA_Scale_Validation.Rmd for full
#    simulation code.
# =============================================================================
df = pd.read_csv("dfGRMSAAW.csv").astype(float)

In [6]:
item_names = list(df.columns)
print(f"\nDataset shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Items         : {item_names[:4]} ... {item_names[-2:]}")
print(f"Value range   : {df.values.min():.0f} – {df.values.max():.0f}")


Dataset shape : 304 rows × 22 columns
Items         : ['AS1', 'AS2', 'AS3', 'AS4'] ... ['AUA3', 'AUA4']
Value range   : 0 – 5


In [7]:
# =============================================================================
# 3. DESCRIPTIVE STATISTICS
# =============================================================================
desc = df.describe().T[["mean", "std", "min", "max"]]
desc.columns = ["Mean", "SD", "Min", "Max"]
desc["Skewness"] = df.skew().round(2)
desc["Kurtosis"] = df.kurtosis().round(2)

In [8]:
print("\nTable 1. Descriptive Statistics for GRMSAAW Items (N = 304)")
print("=" * 60)
print(desc.round(2).to_string())


Table 1. Descriptive Statistics for GRMSAAW Items (N = 304)
      Mean    SD  Min  Max  Skewness  Kurtosis
AS1   2.90  1.22  0.0  5.0     -0.21     -0.45
AS2   3.28  0.84  1.0  5.0      0.07     -0.21
AS3   3.42  1.25  0.0  5.0     -0.45     -0.50
AS4   2.81  1.47  0.0  5.0     -0.23     -0.79
AS5   3.60  1.41  0.0  5.0     -0.69     -0.53
AS6   3.11  0.95  0.0  5.0     -0.13     -0.17
AS7   3.77  0.96  1.0  5.0     -0.56      0.00
AS8   3.04  1.15  0.0  5.0     -0.29     -0.20
AS9   2.87  1.21  0.0  5.0     -0.30     -0.44
AF1   3.25  1.24  0.0  5.0     -0.30     -0.58
AF2   3.52  1.23  0.0  5.0     -0.52     -0.38
AF3   3.17  1.32  0.0  5.0     -0.32     -0.67
AF4   3.15  1.27  0.0  5.0     -0.23     -0.75
MI1   4.15  0.75  2.0  5.0     -0.49     -0.36
MI2   4.51  0.68  2.0  5.0     -1.25      1.03
MI3   4.47  0.72  2.0  5.0     -1.20      0.74
MI4   4.35  0.75  2.0  5.0     -0.90      0.17
MI5   4.61  0.63  2.0  5.0     -1.42      1.23
AUA1  4.24  0.84  1.0  5.0     -0.89      0.23

In [9]:
# =============================================================================
# 4. INTER-ITEM CORRELATION HEATMAP
# =============================================================================
item_order = (
    [f"AS{i}"  for i in range(1, 10)] +
    [f"AF{i}"  for i in range(1, 5)]  +
    [f"MI{i}"  for i in range(1, 6)]  +
    [f"AUA{i}" for i in range(1, 5)]
)
cor_matrix = df[item_order].corr()

In [10]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cor_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, linewidths=0.4,
    annot_kws={"size": 7}, ax=ax
)
ax.set_title(
    "Figure 1: GRMSAAW Inter-Item Correlation Matrix\n"
    "(Items ordered by subscale: AS → AF → MI → AUA)",
    fontsize=13, fontweight="bold", pad=12
)
plt.tight_layout()
plt.savefig("fig1_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nFigure 1 saved: fig1_correlation_heatmap.png")


Figure 1 saved: fig1_correlation_heatmap.png


In [11]:
# =============================================================================
# 5. MODEL 1: UNIDIMENSIONAL CFA
#    NOTE: semopy requires each model equation on a single line —
#    no line continuations. We build the string programmatically.
# =============================================================================
print("\n" + "=" * 60)
print("MODEL 1: UNIDIMENSIONAL CFA")
print("=" * 60)


MODEL 1: UNIDIMENSIONAL CFA


In [12]:
uni_spec = "GRMSAAW =~ " + " + ".join(item_names)

In [13]:
model1 = semopy.Model(uni_spec)
res1   = model1.fit(df)
stats1 = semopy.calc_stats(model1)

In [14]:
print("\nFit Statistics — Unidimensional Model:")
print(stats1.T.round(3).to_string())


Fit Statistics — Unidimensional Model:
                  Value
DoF             209.000
DoF Baseline    231.000
chi2           1004.137
chi2 p-value      0.000
chi2 Baseline  2114.899
CFI               0.578
GFI               0.525
AGFI              0.475
NFI               0.525
TLI               0.534
RMSEA             0.112
AIC              81.394
BIC             244.943
LogLik            3.303


In [15]:
# Extract key indices (stats1 shape: (1, 14); iloc[0] → Series indexed by metric names)
s1 = stats1.iloc[0]
cfi1   = float(s1["CFI"])
rmsea1 = float(s1["RMSEA"])
srmr1  = float(s1["SRMR"]) if "SRMR" in s1.index else np.nan
aic1   = float(s1["AIC"])
bic1   = float(s1["BIC"])
chi1   = float(s1["chi2"])
dof1   = float(s1["DoF"])

In [16]:
# =============================================================================
# 6. MODEL 2: FOUR-FACTOR CORRELATED CFA
#    lavaan-style syntax, one factor per line.
#    By default, semopy correlates all latent variables (oblique model).
# =============================================================================
print("\n" + "=" * 60)
print("MODEL 2: FOUR-FACTOR CORRELATED CFA")
print("=" * 60)


MODEL 2: FOUR-FACTOR CORRELATED CFA


In [17]:
four_spec = "\n".join([
    "AS  =~ " + " + ".join([f"AS{i}"  for i in range(1, 10)]),
    "AF  =~ " + " + ".join([f"AF{i}"  for i in range(1, 5)]),
    "MI  =~ " + " + ".join([f"MI{i}"  for i in range(1, 6)]),
    "AUA =~ " + " + ".join([f"AUA{i}" for i in range(1, 5)]),
])
print("Model syntax:\n", four_spec)

Model syntax:
 AS  =~ AS1 + AS2 + AS3 + AS4 + AS5 + AS6 + AS7 + AS8 + AS9
AF  =~ AF1 + AF2 + AF3 + AF4
MI  =~ MI1 + MI2 + MI3 + MI4 + MI5
AUA =~ AUA1 + AUA2 + AUA3 + AUA4


In [18]:
model4 = semopy.Model(four_spec)
res4   = model4.fit(df)
stats4 = semopy.calc_stats(model4)

In [19]:
print("\nFit Statistics — Four-Factor Model:")
print(stats4.T.round(3).to_string())


Fit Statistics — Four-Factor Model:
                  Value
DoF             203.000
DoF Baseline    231.000
chi2            220.858
chi2 p-value      0.186
chi2 Baseline  2114.899
CFI               0.991
GFI               0.896
AGFI              0.881
NFI               0.896
TLI               0.989
RMSEA             0.017
AIC              98.547
BIC             284.398
LogLik            0.727


In [20]:
s4 = stats4.iloc[0]
cfi4   = float(s4["CFI"])
rmsea4 = float(s4["RMSEA"])
srmr4  = float(s4["SRMR"]) if "SRMR" in s4.index else np.nan
aic4   = float(s4["AIC"])
bic4   = float(s4["BIC"])
chi4   = float(s4["chi2"])
dof4   = float(s4["DoF"])

In [21]:
# --- Factor loadings table ---
# semopy inspect() uses columns: lval, op, rval, Estimate, Std. Err, z-value, p-value
params4 = model4.inspect(mode="list", what="est")

In [22]:
# Factor loadings: op == "~" with lval = item, rval = factor
# (semopy uses reflective notation: item ~ factor)
loadings4 = params4[params4["op"] == "~"].copy()
loadings4 = loadings4[["rval", "lval", "Estimate", "Std. Err", "z-value", "p-value"]].rename(
    columns={"rval": "Factor", "lval": "Item",
             "Std. Err": "SE", "z-value": "z", "p-value": "p"}
)
# Keep only rows where Factor is a latent variable (AS, AF, MI, AUA)
lat_factors = ["AS", "AF", "MI", "AUA"]
loadings4 = loadings4[loadings4["Factor"].isin(lat_factors)]

In [23]:
print("\nTable 2. Factor Loading Estimates — Four-Factor Model:")
print(loadings4.round(3).to_string(index=False))


Table 2. Factor Loading Estimates — Four-Factor Model:
Factor Item  Estimate        SE          z         p
    AS  AS1     1.000         -          -         -
    AS  AS2     0.626  0.047408  13.211547       0.0
    AS  AS3     0.912  0.070439  12.954146       0.0
    AS  AS4     1.084  0.083091  13.047632       0.0
    AS  AS5     0.966  0.080796  11.955275       0.0
    AS  AS6     0.626  0.054981  11.388859       0.0
    AS  AS7     0.658  0.054772   12.00661       0.0
    AS  AS8     0.755  0.066289   11.39403       0.0
    AS  AS9     0.735  0.070529  10.426831       0.0
    AF  AF1     1.000         -          -         -
    AF  AF2     0.824  0.075358  10.934562       0.0
    AF  AF3     0.932  0.081167  11.487011       0.0
    AF  AF4     0.803  0.077399  10.368434       0.0
    MI  MI1     1.000         -          -         -
    MI  MI2     0.848   0.14504   5.847096       0.0
    MI  MI3     0.812  0.145051   5.595676       0.0
    MI  MI4     0.797  0.146536   5.439434 

In [24]:
# =============================================================================
# 7. FACTOR LOADINGS VISUALISATION
# =============================================================================
factor_colors = {"AS": "#2166ac", "AF": "#d73027", "MI": "#4dac26", "AUA": "#d01c8b"}
factor_labels = {
    "AS":  "Ascribed Submissiveness (AS)",
    "AF":  "Asian Fetishism (AF)",
    "MI":  "Media Invalidation (MI)",
    "AUA": "Universal Appearance (AUA)"
}

In [25]:
fig, ax = plt.subplots(figsize=(10, 8))
for _, row in loadings4.iterrows():
    color = factor_colors.get(row["Factor"], "gray")
    ax.barh(row["Item"], row["Estimate"], color=color, alpha=0.82)

In [26]:
ax.axvline(0,   color="black", linewidth=0.8)
ax.axvline(0.3, color="gray",  linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Loading Estimate", fontsize=12)
ax.set_title(
    "Figure 2: Factor Loadings — Four-Factor CFA\n"
    "(dashed line = .30 practical threshold)",
    fontsize=13, fontweight="bold"
)
legend_els = [
    Patch(facecolor=factor_colors[k], label=factor_labels[k])
    for k in lat_factors
]
ax.legend(handles=legend_els, loc="lower right", fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fig2_factor_loadings.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nFigure 2 saved: fig2_factor_loadings.png")


Figure 2 saved: fig2_factor_loadings.png


In [27]:
# =============================================================================
# 8. MODEL COMPARISON
# =============================================================================
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)


MODEL COMPARISON


In [28]:
compare_df = pd.DataFrame({
    "Model":      ["Model 1 (Unidimensional)", "Model 2 (4-Factor)"],
    "χ²":         [round(chi1, 3), round(chi4, 3)],
    "df":         [int(dof1), int(dof4)],
    "CFI":        [round(cfi1, 3), round(cfi4, 3)],
    "RMSEA":      [round(rmsea1, 3), round(rmsea4, 3)],
    "SRMR":       [round(srmr1, 3) if not np.isnan(srmr1) else "-",
                   round(srmr4, 3) if not np.isnan(srmr4) else "-"],
    "AIC":        [round(aic1, 2), round(aic4, 2)],
    "BIC":        [round(bic1, 2), round(bic4, 2)],
}).set_index("Model")

In [29]:
print("\nTable 3. Model Fit Comparison")
print(compare_df.to_string())


Table 3. Model Fit Comparison
                                χ²   df    CFI  RMSEA SRMR    AIC     BIC
Model                                                                    
Model 1 (Unidimensional)  1004.137  209  0.578  0.112    -  81.39  244.94
Model 2 (4-Factor)         220.858  203  0.991  0.017    -  98.55  284.40


In [30]:
# Chi-square difference test
dchi2 = chi1 - chi4
ddf   = dof1 - dof4
pval  = 1 - chi2_dist.cdf(dchi2, ddf)
crit  = chi2_dist.ppf(0.95, ddf)

In [31]:
print(f"\nChi-Square Difference Test:")
print(f"  Δχ²({ddf:.0f}) = {dchi2:.3f},  p = {pval:.6f}")
print(f"  Critical value (α=.05, df={ddf:.0f}): {crit:.3f}")
decision = "Model 2 is significantly superior (p < .001)" if pval < .001 else "No significant difference detected"
print(f"  → {decision}")


Chi-Square Difference Test:
  Δχ²(6) = 783.278,  p = 0.000000
  Critical value (α=.05, df=6): 12.592
  → Model 2 is significantly superior (p < .001)


In [32]:
# =============================================================================
# 9. FIT COMPARISON VISUALISATION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

In [33]:
# Left panel: CFI / RMSEA / SRMR
idx_labels = ["CFI", "RMSEA", "SRMR"]
# Use semopy computed values; fallback to 0 if NaN
srmr1_plot = srmr1 if not np.isnan(srmr1) else 0
srmr4_plot = srmr4 if not np.isnan(srmr4) else 0
m1_vals = [cfi1, rmsea1, srmr1_plot]
m4_vals = [cfi4, rmsea4, srmr4_plot]
x = np.arange(len(idx_labels))
w = 0.35

In [34]:
axes[0].bar(x - w/2, m1_vals, w, label="Model 1 (Unidimensional)", color="#2166ac", alpha=0.8)
axes[0].bar(x + w/2, m4_vals, w, label="Model 2 (4-Factor)",       color="#66c2a5", alpha=0.8)
axes[0].axhline(0.95, linestyle="--", color="#d73027", lw=1.2, label="CFI ≥ .95")
axes[0].axhline(0.10, linestyle=":",  color="gray",   lw=1.2, label="RMSEA/SRMR < .10")
axes[0].set_xticks(x); axes[0].set_xticklabels(idx_labels, fontsize=12)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Index Value", fontsize=11)
axes[0].set_title("Fit Indices\n(higher CFI = better; lower RMSEA/SRMR = better)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8)

In [35]:
# Right panel: AIC / BIC
info_labels = ["AIC", "BIC"]
# Use semopy values (scaled per semopy convention)
m1_info = [aic1, bic1]
m4_info = [aic4, bic4]
x2 = np.arange(len(info_labels))
axes[1].bar(x2 - w/2, m1_info, w, label="Model 1 (Unidimensional)", color="#2166ac", alpha=0.8)
axes[1].bar(x2 + w/2, m4_info, w, label="Model 2 (4-Factor)",       color="#66c2a5", alpha=0.8)
axes[1].set_xticks(x2); axes[1].set_xticklabels(info_labels, fontsize=12)
axes[1].set_ylabel("Criterion Value", fontsize=11)
axes[1].set_title("Information Criteria\n(lower = better fit + parsimony)",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8)

In [36]:
fig.suptitle("Figure 3: CFA Model Comparison — Unidimensional vs. Four-Factor",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig3_model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nFigure 3 saved: fig3_model_comparison.png")


Figure 3 saved: fig3_model_comparison.png


In [37]:
# =============================================================================
# 10. APA RESULTS SUMMARY
# =============================================================================
print(f"""
================================================================
APA-STYLE RESULTS SUMMARY
================================================================

Confirmatory factor analysis was conducted in Python using
semopy {semopy.__version__} (Meshcheryakov et al., 2021)
with maximum likelihood estimation (N = 304).

MODEL 1 — UNIDIMENSIONAL:
  χ²({int(dof1)}) = {chi1:.3f},  p < .001
  CFI = {cfi1:.3f}  |  RMSEA = {rmsea1:.3f}  |  AIC = {aic1:.2f}
  → Poor fit. Single-factor model rejected.
  (CFI well below .95 threshold; RMSEA exceeds .10 danger zone)

MODEL 2 — FOUR-FACTOR CORRELATED:
  χ²({int(dof4)}) = {chi4:.3f},  CFI = {cfi4:.3f}
  RMSEA = {rmsea4:.3f}  |  AIC = {aic4:.2f}  |  BIC = {bic4:.2f}
  → Substantially improved fit on all indices.

MODEL COMPARISON:
  Δχ²({int(ddf)}) = {dchi2:.3f},  p = {pval:.4f}
  ΔAIC = {aic1 - aic4:.2f}  (Model 2 lower — preferred)
  ΔBIC = {bic1 - bic4:.2f}  (Model 2 lower — preferred)
  → Four-factor correlated model is significantly superior.
     The GRMSAAW functions as a four-dimensional instrument.

NOTE: AIC/BIC values differ from R lavaan because semopy uses
a per-observation likelihood scaling. Use R output for
publication-ready AIC/BIC values; use semopy output for
relative within-Python model comparison.
================================================================
""")


APA-STYLE RESULTS SUMMARY

Confirmatory factor analysis was conducted in Python using
semopy 2.3.11 (Meshcheryakov et al., 2021)
with maximum likelihood estimation (N = 304).

MODEL 1 — UNIDIMENSIONAL:
  χ²(209) = 1004.137,  p < .001
  CFI = 0.578  |  RMSEA = 0.112  |  AIC = 81.39
  → Poor fit. Single-factor model rejected.
  (CFI well below .95 threshold; RMSEA exceeds .10 danger zone)

MODEL 2 — FOUR-FACTOR CORRELATED:
  χ²(203) = 220.858,  CFI = 0.991
  RMSEA = 0.017  |  AIC = 98.55  |  BIC = 284.40
  → Substantially improved fit on all indices.

MODEL COMPARISON:
  Δχ²(6) = 783.278,  p = 0.0000
  ΔAIC = -17.15  (Model 2 lower — preferred)
  ΔBIC = -39.46  (Model 2 lower — preferred)
  → Four-factor correlated model is significantly superior.
     The GRMSAAW functions as a four-dimensional instrument.

NOTE: AIC/BIC values differ from R lavaan because semopy uses
a per-observation likelihood scaling. Use R output for
publication-ready AIC/BIC values; use semopy output for
relative